In [1]:
## ====================================
## EET-4501 – Applied Machine Learning
## Project Assignment 4
## ====================================

In [2]:
## Part 2: Dataset Loading & Basic Exploration

## We load the Robotic Operations Performance Dataset from the provided CSV file.
## This dataset contains operational metrics of industrial robotic systems, including processing time, accuracy, energy consumption, sensor readings, and defect detection indicators.

import pandas as pd
import numpy as np
import re

df = pd.read_csv("robot_dataset.csv")

print("🔹 Original Dataset Shape:", df.shape)
display(df.head())

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print("\n🔹 Cleaned Column Names:")
print(df.columns.tolist())

def extract_numeric(value):
    if pd.isnull(value):
        return np.nan
    match = re.search(r"-?\d+\.?\d*", str(value))
    return float(match.group()) if match else np.nan

if "sensor_data" in df.columns:
    df["sensor_value"] = df["sensor_data"].apply(extract_numeric)
    df.drop(columns=["sensor_data"], inplace=True)

print("\n🔹 After Sensor Cleaning:")
print(df.shape)
display(df.head())

binary_columns = [
    "human_intervention_needed",
    "obstacle_detected",
    "defect_detected"
]

for col in binary_columns:
    if col in df.columns:
        df[col] = df[col].map({"Yes": 1, "No": 0})

print("\n🔹 After Binary Encoding:")
display(df.head())

for col in ["robot_id", "component_id"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

print("\n🔹 After Dropping ID Columns:")
print("Final Cleaned Dataset Shape:", df.shape)
display(df.head())

🔹 Original Dataset Shape: (500, 12)


,Robot_ID,Task_Type,Component_ID,Sensor_Type,Sensor_Data,Processing_Time (s),Accuracy (%),Environmental_Status,Energy_Consumption (kWh),Human_Intervention_Needed,Obstacle_Detected,Defect_Detected
0,RBT_001,Inspection,CMP_460,LIDAR,1 (obstacle detected),67.0,90.4,Stable,2.2,No,Yes,Yes
1,RBT_002,Assembly,CMP_252,Thermal,85.3 (°C),71.2,98.1,Stable,2.7,Yes,No,No
2,RBT_003,Inspection,CMP_248,Thermal,92% (visual fit),49.2,95.3,Unstable,2.4,No,No,No
3,RBT_004,Welding,CMP_433,Camera,98% (defect-free),74.5,90.2,Stable,2.4,Yes,No,Yes
4,RBT_005,Assembly,CMP_992,Camera,92% (visual fit),64.5,97.2,Unstable,1.8,No,No,No



🔹 Cleaned Column Names:
['robot_id', 'task_type', 'component_id', 'sensor_type', 'sensor_data', 'processing_time_(s)', 'accuracy_(%)', 'environmental_status', 'energy_consumption_(kwh)', 'human_intervention_needed', 'obstacle_detected', 'defect_detected']

🔹 After Sensor Cleaning:
(500, 12)


,robot_id,task_type,component_id,sensor_type,processing_time_(s),accuracy_(%),environmental_status,energy_consumption_(kwh),human_intervention_needed,obstacle_detected,defect_detected,sensor_value
0,RBT_001,Inspection,CMP_460,LIDAR,67.0,90.4,Stable,2.2,No,Yes,Yes,1.0
1,RBT_002,Assembly,CMP_252,Thermal,71.2,98.1,Stable,2.7,Yes,No,No,85.3
2,RBT_003,Inspection,CMP_248,Thermal,49.2,95.3,Unstable,2.4,No,No,No,92.0
3,RBT_004,Welding,CMP_433,Camera,74.5,90.2,Stable,2.4,Yes,No,Yes,98.0
4,RBT_005,Assembly,CMP_992,Camera,64.5,97.2,Unstable,1.8,No,No,No,92.0



🔹 After Binary Encoding:


,robot_id,task_type,component_id,sensor_type,processing_time_(s),accuracy_(%),environmental_status,energy_consumption_(kwh),human_intervention_needed,obstacle_detected,defect_detected,sensor_value
0,RBT_001,Inspection,CMP_460,LIDAR,67.0,90.4,Stable,2.2,0,1,1,1.0
1,RBT_002,Assembly,CMP_252,Thermal,71.2,98.1,Stable,2.7,1,0,0,85.3
2,RBT_003,Inspection,CMP_248,Thermal,49.2,95.3,Unstable,2.4,0,0,0,92.0
3,RBT_004,Welding,CMP_433,Camera,74.5,90.2,Stable,2.4,1,0,1,98.0
4,RBT_005,Assembly,CMP_992,Camera,64.5,97.2,Unstable,1.8,0,0,0,92.0



🔹 After Dropping ID Columns:
Final Cleaned Dataset Shape: (500, 10)


,task_type,sensor_type,processing_time_(s),accuracy_(%),environmental_status,energy_consumption_(kwh),human_intervention_needed,obstacle_detected,defect_detected,sensor_value
0,Inspection,LIDAR,67.0,90.4,Stable,2.2,0,1,1,1.0
1,Assembly,Thermal,71.2,98.1,Stable,2.7,1,0,0,85.3
2,Inspection,Thermal,49.2,95.3,Unstable,2.4,0,0,0,92.0
3,Welding,Camera,74.5,90.2,Stable,2.4,1,0,1,98.0
4,Assembly,Camera,64.5,97.2,Unstable,1.8,0,0,0,92.0


In [3]:
## Part 3: Feature Typing & Preprocessing

## We categorize features into numerical and categorical types to build a structured preprocessing pipeline.

y = df["human_intervention_needed"]

X = df.drop(columns=["human_intervention_needed"])

print("\n🔹 Feature Matrix Shape:", X.shape)
print("🔹 Target Vector Shape:", y.shape)

print("\n🔹 Feature Preview:")
display(X.head())

print("\n🔹 Target Preview:")
display(y.head())

if "accuracy" in X.columns and "processing_time_(s)" in X.columns:
    X["efficiency_score"] = X["accuracy"] / X["processing_time_(s)"]

if "accuracy" in X.columns and "energy_consumption_(kwh)" in X.columns:
    X["energy_efficiency"] = X["accuracy"] / X["energy_consumption_(kwh)"]

risk_cols = [
    col for col in ["obstacle_detected", "defect_detected"]
    if col in X.columns
]

if risk_cols:
    X["risk_flag"] = X[risk_cols].sum(axis=1)

print("\n🔹 After Feature Engineering:")
display(X.head())

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n🔹 Train/Test Split Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


🔹 Feature Matrix Shape: (500, 9)
🔹 Target Vector Shape: (500,)

🔹 Feature Preview:


,task_type,sensor_type,processing_time_(s),accuracy_(%),environmental_status,energy_consumption_(kwh),obstacle_detected,defect_detected,sensor_value
0,Inspection,LIDAR,67.0,90.4,Stable,2.2,1,1,1.0
1,Assembly,Thermal,71.2,98.1,Stable,2.7,0,0,85.3
2,Inspection,Thermal,49.2,95.3,Unstable,2.4,0,0,92.0
3,Welding,Camera,74.5,90.2,Stable,2.4,0,1,98.0
4,Assembly,Camera,64.5,97.2,Unstable,1.8,0,0,92.0



🔹 Target Preview:


0    0
1    1
2    0
3    1
4    0
Name: human_intervention_needed, dtype: int64


🔹 After Feature Engineering:


,task_type,sensor_type,processing_time_(s),accuracy_(%),environmental_status,energy_consumption_(kwh),obstacle_detected,defect_detected,sensor_value,risk_flag
0,Inspection,LIDAR,67.0,90.4,Stable,2.2,1,1,1.0,2
1,Assembly,Thermal,71.2,98.1,Stable,2.7,0,0,85.3,0
2,Inspection,Thermal,49.2,95.3,Unstable,2.4,0,0,92.0,0
3,Welding,Camera,74.5,90.2,Stable,2.4,0,1,98.0,1
4,Assembly,Camera,64.5,97.2,Unstable,1.8,0,0,92.0,0



🔹 Train/Test Split Shapes:
X_train: (400, 10)
X_test: (100, 10)
y_train: (400,)
y_test: (100,)
